# 02 期待値との残差で検知する — 「天気で説明できない」低下を見つける

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 天気から「正常ならこのくらい」を予想し、実際との差（残差）で異常を見つける回です。
> - **たとえ話**: 雨の日にお客が少ないのは当たり前。「天気のわりに少ない日」だけを問題にする。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-4（天気を考慮する・残差）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

## ゴール
- 日射量などの天気の情報から**期待される発電量**を予測し、実測との**残差**で異常を検知する。
- 01（天気を考えない方法）と比べて、**何が改善し、何が改善しないか**を確かめる。
- 「平滑化（直近の平均）」が、徐々に進む異常に効く一方で、**警報の居残り（余韻）**を生むことを理解する。

## 前のNotebookとのつながり
01 では、天気を考えない方法は停電しか見つけられませんでした。雨の日の低下は正常なのに、それを区別できないためです。
ここでは「**天気から期待される量より少ない**」ことを異常と考えます。

## やること
1. 正常データ（train）で「期待発電量」のモデルを学習する
2. 実測との残差（相対残差）を見る
3. しきい値を、学習に使っていない年の残差で決める理由を確かめる
4. 3種類の期待値モデル × 平滑化なし/7日平均 を、10系列で比べる

## キーワード
- **期待発電量**: 天気の情報（日射量など）から予測した、「正常ならこのくらい」の発電量。
- **残差**: 期待値と実測の差。ここでは **相対残差 =（期待値 − 実測）÷ 期待値**。大きいほど「期待より少ない」。
- **ホールドアウト（out-of-fold）**: 学習に使わなかったデータで評価すること。
- **過学習**: 学習に使ったデータには合うが、新しいデータには合わなくなること。
- **平滑化**: 直近数日の平均を取ること。ノイズが減り、徐々に進む変化が見えやすくなる。
- **余韻**: 平滑化のせいで、異常が終わった後も数日、警報が出続けること。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from anomaly_synth import make_split
from anomaly_eval import alarm_by_normal_quantile
from baselines import DETECTOR_COLUMNS
from residual import FEATURES, fit_expected, relative_residual, out_of_fold_residual, smooth, make_features
import run_phase2

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
train = make_split("train")[DETECTOR_COLUMNS]      # 正常のみ
val = make_split("val", 1)                         # 異常あり（評価用）
x = val[DETECTOR_COLUMNS]

model = fit_expected(train, "linear_gsr")          # 日射量だけから発電量を予測する直線
val_expected = model.predict(make_features(x, "linear_gsr"))
print("傾き（kWh / MJ/m²）:", round(model.coef_[0], 3), " 切片:", round(model.intercept_, 3))

**何をしているか**: 正常データだけで、「日射量から発電量を予測する直線」を学習します。

**出力の見方**: 傾きは約0.85です。データを作ったときの換算係数（0.85）とほぼ一致しています。
正常データだけで、天気と発電量の関係を学べました。

In [ ]:
resid = relative_residual(x, model, "linear_gsr")
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(val["date"], val["gen_kwh"], color="gray", lw=0.8, label="実測")
ax[0].plot(val["date"], val_expected, color="tab:blue", lw=0.8, alpha=0.7, label="期待値（天気から予測）")
ax[0].legend(ncol=2); ax[0].set(ylabel="kWh", title="実測と期待値")
colors = {"outage": "red", "degrade": "orange", "shade": "purple", "stuck": "green"}
ax[1].plot(val["date"], resid, color="gray", lw=0.8)
for kind, c in colors.items():
    d = val[val["anomaly_type"] == kind]
    ax[1].scatter(d["date"], resid[d.index], s=14, color=c, label=kind, zorder=3)
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set(ylabel="相対残差", title="相対残差（大きいほど期待より少ない）"); ax[1].legend(ncol=4)
fig.tight_layout()

**何をしているか**: 上は実測と期待値、下はその相対残差です。色付きの点が異常日です。

**出力の見方**:
- 上の図では、雨の日に実測が大きく下がっても、期待値も同じように下がっています。**天気による上下は、期待値が説明してくれます。**
- 下の図では、正常日の残差はほぼ0の周りに収まり、**停電（赤）と影（紫）は大きく上に外れます**。01 では見分けられなかった異常が、はっきり見えます。
- 劣化（橙）は、徐々に上へ上がっていきます。序盤は正常のばらつきの範囲です。
- 固着（緑）は、天気が変わるのに値が動かないので、残差が上にも下にも振れます。

In [ ]:
rows = []
for kind in FEATURES:
    m = fit_expected(train, kind)
    rows.append(dict(期待値モデル=kind,
                     学習に使った年の残差のばらつき=round(relative_residual(train, m, kind).std(), 4),
                     学習に使っていない年の残差のばらつき=round(out_of_fold_residual(train, kind).std(), 4)))
pd.DataFrame(rows)

**何をしているか**: 3種類の期待値モデルについて、正常データの残差のばらつき（標準偏差）を2通りで測ります。
- 学習に使った年の残差（モデルが答えを知っている）
- 学習に使っていない年の残差（年ごとにモデルを作り直して、その年を除いて学習したもの）

**出力の見方**:
- 線形モデルは、2つの値がほぼ同じです（約0.056）。
- RandomForest は、学習に使った年は**0.052と小さく**、使っていない年は**0.071と大きく**なります。**過学習**です。
- もし学習に使った年の残差でしきい値を決めると、RandomForest は「残差が小さいのが普通」と思い込み、新しいデータで**誤警報が増えます**。
  このプロジェクトでは、しきい値を**学習に使っていない年の残差**で決めています。
- なお、この合成データは日射量とほぼ比例する単純な関係なので、複雑な RandomForest の出番はありません。

In [ ]:
res, agg, chance = run_phase2.run(verbose=False)
agg

**何をしているか**: 6つの検知器を、val の10系列で評価します（しきい値は、学習に使っていない年の残差の上位3%）。
01 と同じ評価条件です。

**出力の見方（検知器ごと）**:
- **日次**の3つは、誤警報が月0.8〜1.1回で、`precision`（警報の適合率）は0.73〜0.78です。01 の方法（0.45〜0.6）より高くなっています。
- `auprc` は、日次の3つが0.67〜0.69で、01 の0.32〜0.35の約2倍です。3つの間の差は標準偏差（0.06前後）の範囲内で、**差があるとは言えません**。
- **7日平均**の3つは、`fa_mean` が月4.4回と急に増えます。ただし、`fa_grace7`（異常が終わった直後7日間の正常日を除いた誤警報）は月0.5回前後です。
  これが、平滑化の**余韻**です。次のセルで確かめます。

In [ ]:
print("== 検知率（イベント単位） ==")
print(res.pivot(index="detector", columns="type", values="recall").loc[agg.index])
print("\n== 日単位の検知率 ==")
print(res.pivot(index="detector", columns="type", values="day_recall").loc[agg.index])
print("\n== 検知までの日数 ==")
print(res.pivot(index="detector", columns="type", values="delay_mean").loc[agg.index])
print("\n偶然の水準:", chance.to_dict())

**出力の見方**:
- **停電・影**: 日次で、イベント単位も日単位も1.0です。01（影は0.3）から大きく改善し、**天気を考えると影も確実に見つかる**ことが分かります。
- **劣化**: イベント単位は1.0で、偶然の水準0.64を明確に上回ります。日単位は日次で0.29〜0.30、7日平均で0.52〜0.58です。
  検知までは日次で約7日、7日平均で約11〜13日かかります。**徐々に進む異常は、数日分の平均を取ると、日単位では見つけやすくなります**。ただし、見つかるのは悪化が進んだ後半で、序盤の小さな低下は見逃します。
- **固着**: 日次で0.55（日単位は0.35〜0.37）にとどまり、7日平均ではかえって悪化します（0.30〜0.35）。
  固着は「値が動かない」異常なので、**残差では検知しきれません**。01 の固着ルール（同じ値が続く：1.0）が必要です。
- **異常の種類によって、向く方法が違う**ので、最終的には組み合わせて使うことになります。

In [ ]:
model = fit_expected(train, "linear_gsr")
thr_daily = alarm_by_normal_quantile(out_of_fold_residual(train, "linear_gsr"), 0.03)
thr_7d = alarm_by_normal_quantile(smooth(out_of_fold_residual(train, "linear_gsr"), 7), 0.03)
s1 = relative_residual(x, model, "linear_gsr")
s7 = smooth(s1, 7)

# 停電（outage）の最初のイベントの前後を拡大する
first = val[val["anomaly_type"] == "outage"].index[0]
sl = slice(first - 4, first + 18)
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for a, s, thr, title in [(ax[0], s1, thr_daily, "日次の相対残差"), (ax[1], s7, thr_7d, "7日平均の相対残差")]:
    a.plot(val["date"][sl], s[sl], marker="o", ms=3, color="gray")
    a.axhline(thr, color="red", ls="--", lw=1, label="しきい値")
    bad = val["is_anomaly"][sl] == 1
    a.scatter(val["date"][sl][bad], s[sl][bad], color="red", s=25, zorder=3, label="異常日")
    a.set(title=title); a.legend()
fig.tight_layout()

**何をしているか**: 停電の前後を拡大して、日次の残差（上）と7日平均の残差（下）を並べます。

**出力の見方**:
- 上（日次）: 停電の日だけ、しきい値を超えます。停電が終われば、すぐに戻ります。
- 下（7日平均）: 停電が終わったあとも、直近7日の平均に停電の日が残るため、**数日間しきい値を超えたまま**です。
  これが「余韻」で、正常な日に出る警報なので、評価上は誤警報に数えられてしまいます。
- 実際の運用では、「異常が終わった直後の数日は警報が続く」と分かっていれば、許容できることもあります。
  このため、余韻を除いた誤警報（`fa_grace7`）も別に報告しています。

## 試してみよう（演習）
1. **学習に使った年でしきい値を決めるとどうなるか**: 残差のばらつきを表にしたセルの `out_of_fold_residual(train, kind)` を `relative_residual(train, fit_expected(train, kind), kind)` に置き換えて、RandomForest の残差のばらつきを比べてください。
   見るポイントは0.052と0.071の違いです。分かるのは、モデルが答えを知っているデータでしきい値を決めると、しきい値が厳しすぎて、新しいデータで誤警報が増えるということです。
2. **平滑化の窓を変える**: `src/run_phase2.py` の `CONFIGS` にある `7` を `3` と `14` にして、`python src/run_phase2.py` を実行してください。
   見るポイントは、劣化の日単位の検知率と、`fa_grace7` と `fa_mean` の差です。分かるのは、窓を長くすると劣化は見つけやすくなるが、余韻も長くなるという関係です。
3. **誤警報の目標を変える**: `run_phase2.run(rate=0.01)` と `run_phase2.run(rate=0.10)` を実行し、劣化と固着の検知率がどう動くか見てください。分かるのは、誤警報を減らすと、弱い異常から見逃していくことです。

## まとめ
- **天気から期待値を作って残差を見る**と、停電・影は確実に、劣化も偶然を超えて見つけられた（01では不可能）。
- 期待値モデルは、複雑にしても効果はなかった。残差の大きさは**データのノイズ（発電約3%＋日射計測約4%）**で決まるため、単純な直線で十分。
- 複雑なモデルは過学習しやすいので、しきい値は**学習に使っていないデータ**の残差で決める。
- 平滑化は、徐々に進む劣化に効くが、**余韻**で誤警報が増えて見える。固着は残差では見つけにくく、専用ルールが必要。

次の Phase3 では、期待値モデルを作らずに、**教師なし手法**（Isolation Forest・LOF）で検知し、残差法と比べます。